# 00 · Verificar el clúster Spark

Este notebook comprueba que tu clúster **Apache Spark** (1 master + 2 workers) está
funcionando y que puedes ejecutar trabajos distribuidos.

**Antes de empezar:** cuando VS Code te pregunte por el *kernel*, elige el Python
del contenedor (`/usr/bin/python3` o "Python 3").

Los interfaces web del clúster están en la pestaña **PORTS** de VS Code:
- **8080** → Spark Master UI (verás los 2 workers registrados)
- **4040** → Spark Application UI (aparece al lanzar un trabajo)


## 1. Crear la sesión de Spark conectada al clúster

Nos conectamos al *master* en modo **standalone** (`spark://spark-master:7077`).
El notebook actúa como **driver**; los cálculos se reparten en los **workers**.


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("00-verificar-cluster")
    # Conexión al clúster real (no local[*])
    .master("spark://spark-master:7077")
    # El driver corre en este contenedor: los executors deben poder alcanzarlo
    .config("spark.driver.host", "spark-driver")
    .config("spark.driver.bindAddress", "0.0.0.0")
    # Recursos por executor (ajustados a una máquina de 2 nucleos)
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "2")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Version de Spark :", spark.version)
print("Master           :", spark.sparkContext.master)
print("Spark App UI      :", spark.sparkContext.uiWebUrl)
spark

## 2. Ejecutar un cálculo distribuido

Estimamos el número π por el método de Montecarlo. El trabajo se reparte entre
los workers: míralo en directo en la **Master UI (8080)** y en la **App UI (4040)**.


In [ ]:
import random

def dentro_del_circulo(_):
    x, y = random.random(), random.random()
    return 1 if x * x + y * y <= 1 else 0

NUM_MUESTRAS = 20_000_000
conteo = (
    spark.sparkContext
    .parallelize(range(NUM_MUESTRAS), numSlices=16)
    .map(dentro_del_circulo)
    .reduce(lambda a, b: a + b)
)

pi = 4.0 * conteo / NUM_MUESTRAS
print(f"π ≈ {pi}")

## 3. Ver los workers activos

Comprobamos que hay más de un executor trabajando (prueba de que es un clúster
real y no una ejecución local).


In [ ]:
sc = spark.sparkContext
executors = sc._jsc.sc().statusTracker().getExecutorInfos()
print("Executors registrados (incluye el driver):", len(executors))
for e in executors:
    print(" -", e.host())

In [ ]:
# Una pequeña operacion con DataFrame para confirmar Spark SQL
df = spark.range(0, 1_000_000)
print("Filas:", df.count())
df.selectExpr("sum(id) as suma", "avg(id) as media").show()

## 4. (Opcional) Liberar recursos

Al terminar la sesión puedes cerrar Spark. Si vas a seguir con el notebook `01`,
**no** ejecutes esta celda: reutiliza la misma sesión o abre una nueva allí.


In [ ]:
# spark.stop()